In [4]:
import os
import json
import re
import joblib
import requests
import pandas as pd

from jsonschema import validate, ValidationError

In [5]:
model = joblib.load("best_model.pkl")

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [6]:
os.environ["LLM_API_KEY"] = "YOUR_OPENROUTER_API_KEY"

api_key = os.getenv("LLM_API_KEY")

In [7]:
url="https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt,user_prompt,temperature=0,max_tokens=512):

    headers={
        "Authorization":f"Bearer {api_key}",
        "Content-Type":"application/json"
    }

    payload={
        "model":"openai/gpt-4o-mini",
        "messages":[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        "temperature":temperature,
        "max_tokens":max_tokens
    }

    response=requests.post(url,headers=headers,json=payload)

    if response.status_code!=200:
        print(response.status_code)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [8]:
def has_pii(text):

    email=r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone=r'\b\d{10}\b'

    return bool(re.search(email,text) or re.search(phone,text))

In [9]:
schema={

"type":"object",

"properties":{

"prediction_label":{"type":"string"},

"confidence_level":{"type":"string"},

"top_reason":{"type":"string"},

"second_reason":{"type":"string"},

"next_step":{"type":"string"}

},

"required":[

"prediction_label",

"confidence_level",

"top_reason",

"second_reason",

"next_step"

]

}

In [14]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

# Create features exactly like Part 3
y_clf = (df["charges"] > df["charges"].median()).astype(int)

X = df.drop("charges", axis=1)

categorical_columns = X.select_dtypes(include=["object", "category"]).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

print(X.head())

   age     bmi  children  sex_male  smoker_yes  region_northwest  \
0   19  27.900         0     False        True             False   
1   18  33.770         1      True       False             False   
2   28  33.000         3      True       False             False   
3   33  22.705         0      True       False              True   
4   32  28.880         0      True       False              True   

   region_southeast  region_southwest  
0             False              True  
1              True             False  
2              True             False  
3             False             False  
4             False             False  


In [15]:
samples=X.iloc[:3]

for i,row in samples.iterrows():

    pred=model.predict(pd.DataFrame([row]))[0]

    prob=model.predict_proba(pd.DataFrame([row]))[0].max()

    prompt=f"""

Features:

{row.to_dict()}

Prediction:{pred}

Probability:{prob}

Return ONLY JSON.

"""

    if has_pii(prompt):

        print("Blocked")

        continue

    response=call_llm(

        "You are an AI assistant. Output only valid JSON.",

        prompt,

        temperature=0

    )

    print(response)

    try:

        data=json.loads(response)

        validate(data,schema)

        print("Validation Passed")

    except Exception as e:

        print("Validation Failed",e)

401
None
Validation Failed the JSON object must be str, bytes or bytearray, not NoneType
401
None
Validation Failed the JSON object must be str, bytes or bytearray, not NoneType
401
None
Validation Failed the JSON object must be str, bytes or bytearray, not NoneType


In [16]:
# Create dataset again
df = pd.read_csv("cleaned_data.csv")

X = df.drop("charges", axis=1)

categorical_columns = X.select_dtypes(include=["object","category"]).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

samples = X.iloc[:3]

system_prompt = """
You are an AI assistant.
Return ONLY valid JSON with the following fields:

{
"prediction_label":"",
"confidence_level":"",
"top_reason":"",
"second_reason":"",
"next_step":""
}
"""

for i, row in samples.iterrows():

    pred = model.predict(pd.DataFrame([row]))[0]
    prob = model.predict_proba(pd.DataFrame([row]))[0].max()

    user_prompt = f"""
Features:
{row.to_dict()}

Predicted Class: {pred}

Probability: {prob}
"""

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        continue

    response = call_llm(system_prompt, user_prompt, temperature=0)

    print("\nInput", i + 1)
    print(response)

    try:
        data = json.loads(response.strip())
        validate(data, schema)
        print("✅ Validation Passed")
    except Exception as e:
        print("❌ Validation Failed:", e)

401

Input 1
None
❌ Validation Failed: 'NoneType' object has no attribute 'strip'
401

Input 2
None
❌ Validation Failed: 'NoneType' object has no attribute 'strip'
401

Input 3
None
❌ Validation Failed: 'NoneType' object has no attribute 'strip'


In [17]:
print(has_pii("lakshana@gmail.com"))
print(has_pii("Age=45, BMI=28"))

True
False


In [18]:
response0 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

response07 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.7
)

print("Temperature 0")
print(response0)

print("\nTemperature 0.7")
print(response07)

401
401
Temperature 0
None

Temperature 0.7
None
